# Neural Network + DeepFool Pipeline

This notebook follows the handwritten pipeline and uses the model settings loaded from `MachineLearning\NeuralNetworks\best_params.csv`.

Pipeline steps:

1. `clean.fit(X_train, y_train)` to train the clean model  
2. `clean.predict(X_test)` and evaluate the clean model on clean test data  
3. Save the clean model  
4. Initialize the DeepFool attack  
5. Generate adversarial train and test samples  
6. Keep adversarial labels aligned with the original ground-truth labels  
7. Build combined clean+adversarial train and test sets  
8. Retrain using ART's `AdversarialTrainer` with the DeepFool attack  
9. Evaluate both the clean model and the adversarially trained model on:
   - clean test data
   - adversarial test data
   - combined clean+adversarial test data


In [1]:
# If needed, install once:
# !pip install tensorflow scikit-learn adversarial-robustness-toolbox pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf

from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import DeepFool
from art.defences.trainer import AdversarialTrainer


In [2]:
def normalize_path(path_like) -> Path:
    return Path(str(path_like).replace("\\", "/")).expanduser()


def resolve_existing_path(*candidates, required: bool = True):
    checked = []
    for candidate in candidates:
        if candidate is None:
            continue
        p = normalize_path(candidate)
        variants = [p]
        if not p.is_absolute():
            variants.extend([
                Path.cwd() / p,
                Path.cwd().parent / p,
                Path.cwd().parent.parent / p,
            ])
        for variant in variants:
            checked.append(str(variant))
            if variant.exists():
                print(f"Resolved path: {variant}")
                return variant
    if required:
        raise FileNotFoundError("Could not find any of these paths:\n" + "\n".join(checked))
    return None

BEST_PARAMS_PATH = resolve_existing_path(
    Path("../../MachineLearning/NeuralNetworks/best_params.csv"),
    Path("../MachineLearning/NeuralNetworks/best_params.csv"),
    Path("MachineLearning/NeuralNetworks/best_params.csv"),
)

def parse_hidden_layers(value):
    if isinstance(value, (tuple, list)):
        return tuple(int(v) for v in value)

    text = str(value).strip().strip('"').strip("'")
    if text.startswith("(") or text.startswith("["):
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (tuple, list)):
            return tuple(int(v) for v in parsed)

    return tuple(int(part.strip()) for part in text.split(",") if part.strip())

best_params_df = pd.read_csv(BEST_PARAMS_PATH)
BEST_PARAMS = best_params_df.iloc[0].to_dict()
BEST_PARAMS["hidden_layer_sizes"] = parse_hidden_layers(BEST_PARAMS["hidden_layer_sizes"])
BEST_PARAMS["alpha"] = float(BEST_PARAMS["alpha"])
BEST_PARAMS["learning_rate_init"] = float(BEST_PARAMS["learning_rate_init"])
BEST_PARAMS["batch_size"] = int(BEST_PARAMS["batch_size"])
BEST_PARAMS["max_iter"] = int(BEST_PARAMS["max_iter"])
BEST_PARAMS["early_stopping"] = str(BEST_PARAMS["early_stopping"]).lower() in {"true", "1", "yes"}
BEST_PARAMS["n_iter_no_change"] = int(BEST_PARAMS["n_iter_no_change"])
BEST_PARAMS["random_state"] = int(BEST_PARAMS["random_state"])

SEED = BEST_PARAMS["random_state"]
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DEFAULT_DATA_PATH = resolve_existing_path(
    Path("../../CSVs/newDataset.csv"),
    Path("../../CSVs/dataset.csv"),
    Path("../CSVs/newDataset.csv"),
    Path("../CSVs/dataset.csv"),
    Path("CSVs/newDataset.csv"),
    Path("CSVs/dataset.csv"),
    required=False,
)
RUNS_DIR = resolve_existing_path(
    Path("../../StandardizedRuns"),
    Path("../StandardizedRuns"),
    Path("StandardizedRuns"),
    required=False,
)
RUN_GLOB = "NeuralNet_train_*.csv"

# Optional environment overrides:
# - NN_RUN_PATH
# - MODEL_RUN_PATH
ENV_RUN_PATH = os.environ.get("NN_RUN_PATH") or os.environ.get("MODEL_RUN_PATH")

LABEL_COL = "anomaly"
DROP_COLS = {LABEL_COL, "segment", "train", "sampling", "channel"}

TEST_SIZE = 0.75
BATCH_SIZE = BEST_PARAMS["batch_size"]
NB_EPOCHS = BEST_PARAMS["max_iter"]
LR = BEST_PARAMS["learning_rate_init"]
WEIGHT_DECAY = BEST_PARAMS["alpha"]
HIDDEN_LAYER_SIZES = BEST_PARAMS["hidden_layer_sizes"]
ACTIVATION_NAME = str(BEST_PARAMS["activation"]).lower()
SOLVER_NAME = str(BEST_PARAMS["solver"]).lower()
LR_POLICY = str(BEST_PARAMS["learning_rate"]).lower()
EARLY_STOPPING = BEST_PARAMS["early_stopping"]
N_ITER_NO_CHANGE = BEST_PARAMS["n_iter_no_change"]

# DeepFool settings
DEEPFOOL_MAX_ITER = 50
DEEPFOOL_EPSILON = 1e-6
DEEPFOOL_NB_GRADS = 2
DEEPFOOL_BATCH_SIZE = BATCH_SIZE

ADV_RATIO = 0.55

SAVE_MODELS = False
ARTIFACT_DIR = Path("artifacts")
RESULTS_DIR = Path("Results") / "NeuralNetworksResults"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_path() -> Path:
    if ENV_RUN_PATH:
        candidate = normalize_path(ENV_RUN_PATH)
        if candidate.exists():
            return candidate
        print(f"[warn] Env path not found: {candidate}")

    if RUNS_DIR and RUNS_DIR.exists():
        candidates = sorted(
            RUNS_DIR.glob(RUN_GLOB),
            key=lambda x: x.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            return candidates[0]

    if DEFAULT_DATA_PATH and DEFAULT_DATA_PATH.exists():
        return DEFAULT_DATA_PATH

    raise FileNotFoundError("No NN data source found. Check CSVs or set NN_RUN_PATH.")

DATA_PATH = resolve_data_path()
print(f"Using data source: {DATA_PATH}")
print("Loaded best params:")
print(BEST_PARAMS)


Resolved path: ..\..\MachineLearning\NeuralNetworks\best_params.csv
Resolved path: ..\..\CSVs\newDataset.csv
Resolved path: ..\..\StandardizedRuns
Using data source: ..\..\StandardizedRuns\NeuralNet_train_advtrain_round2.csv
Loaded best params:
{'hidden_layer_sizes': (128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.01, 'batch_size': 64, 'activation': 'relu', 'solver': 'adam', 'learning_rate': 'adaptive', 'max_iter': 1000, 'early_stopping': True, 'n_iter_no_change': 20, 'random_state': 100}


In [3]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_df = df[[c for c in df.columns if c not in DROP_COLS]].copy()
    non_numeric_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric feature columns found in {csv_path}: {non_numeric_cols}. "
            "Add them to DROP_COLS or encode them before training."
        )
    feature_cols = feature_df.columns.tolist()
    X = feature_df.to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), feature_cols, scaler

X_train, X_test, y_train, y_test, feature_cols, scaler = load_and_prepare(str(DATA_PATH))

def make_balanced_training_set(X: np.ndarray, y: np.ndarray, seed: int = SEED):
    """Randomly oversample the minority class so the NN does not collapse to only class 0."""
    y_int = y.astype(int)
    classes, counts = np.unique(y_int, return_counts=True)
    if len(classes) != 2:
        raise ValueError(f"Expected two classes for balancing, got {classes}")
    max_count = int(counts.max())
    rng = np.random.default_rng(seed)
    sampled_indices = []
    for cls in classes:
        cls_indices = np.flatnonzero(y_int == cls)
        sampled_indices.append(rng.choice(cls_indices, size=max_count, replace=len(cls_indices) < max_count))
    sampled_indices = np.concatenate(sampled_indices)
    rng.shuffle(sampled_indices)
    X_balanced = X[sampled_indices].astype(np.float32)
    y_balanced = y_int[sampled_indices].astype(np.int64)
    print("Original train label dist:", dict(zip(classes.tolist(), counts.tolist())))
    print("Balanced train label dist:", dict(zip(*np.unique(y_balanced, return_counts=True))))
    return X_balanced, y_balanced

X_train_balanced, y_train_balanced = make_balanced_training_set(X_train, y_train)


y_train_balanced_oh = tf.keras.utils.to_categorical(y_train_balanced, num_classes=2).astype(np.float32)


Loaded: ..\..\StandardizedRuns\NeuralNet_train_advtrain_round2.csv
Rows=2698, Features=18, Label dist=[2150  548]
Train=(674, 18), Test=(2024, 18)
Original train label dist: {0: 537, 1: 137}
Balanced train label dist: {np.int64(0): np.int64(537), np.int64(1): np.int64(537)}


In [4]:
def get_activation(name: str):
    name = name.lower()
    if name == "relu":
        return "relu"
    if name == "tanh":
        return "tanh"
    if name == "logistic":
        return "sigmoid"
    raise ValueError(f"Unsupported activation for this notebook: {name}")


def build_mlp_model(
    d_in: int,
    hidden_layer_sizes=HIDDEN_LAYER_SIZES,
    activation_name: str = ACTIVATION_NAME,
    weight_decay: float = WEIGHT_DECAY,
):
    activation = get_activation(activation_name)
    regularizer = tf.keras.regularizers.l2(weight_decay) if weight_decay else None

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(d_in,)))
    for hidden_units in hidden_layer_sizes:
        model.add(tf.keras.layers.Dense(
            int(hidden_units),
            activation=activation,
            kernel_regularizer=regularizer,
        ))
    model.add(tf.keras.layers.Dense(2, activation="softmax"))
    return model


def make_art_classifier(
    d_in: int,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    hidden_layer_sizes=HIDDEN_LAYER_SIZES,
    activation_name: str = ACTIVATION_NAME,
):
    model = build_mlp_model(
        d_in=d_in,
        hidden_layer_sizes=hidden_layer_sizes,
        activation_name=activation_name,
        weight_decay=weight_decay,
    )
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    loss_object = tf.keras.losses.CategoricalCrossentropy()

    @tf.function
    def train_step(model_instance, x_batch, y_batch):
        with tf.GradientTape() as tape:
            predictions = model_instance(x_batch, training=True)
            loss = loss_object(y_batch, predictions)
            if model_instance.losses:
                loss += tf.add_n(model_instance.losses)
        gradients = tape.gradient(loss, model_instance.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model_instance.trainable_variables))

    return TensorFlowV2Classifier(
        model=model,
        nb_classes=2,
        input_shape=(d_in,),
        loss_object=loss_object,
        train_step=train_step,
        clip_values=(0.0, 1.0),
    )


def predict_labels(art_clf, X: np.ndarray):
    probs = art_clf.predict(X).astype(np.float32)
    return np.argmax(probs, axis=1)


def eval_from_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }


def eval_classifier(art_clf, X: np.ndarray, y: np.ndarray, name: str):
    y_pred = predict_labels(art_clf, X)
    metrics = eval_from_predictions(y, y_pred, name)
    return metrics, y_pred


def save_art_model_state(art_clf, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    art_clf.model.save_weights(str(out_path))
    print(f"Saved model weights: {out_path}")


In [5]:
# Step 1: clean.fit(X_train, y_train) -> trained clean model
print('Training clean model with:', {
    'hidden_layer_sizes': HIDDEN_LAYER_SIZES,
    'alpha': WEIGHT_DECAY,
    'learning_rate_init': LR,
    'batch_size': BATCH_SIZE,
    'activation': ACTIVATION_NAME,
    'solver': SOLVER_NAME,
    'learning_rate': LR_POLICY,
    'max_iter': NB_EPOCHS,
    'early_stopping': EARLY_STOPPING,
    'n_iter_no_change': N_ITER_NO_CHANGE,
    'random_state': SEED,
})

art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train_balanced, y_train_balanced_oh, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

# y_pred = clean.predict(X_test)
# eval_classifier(clean, X_test, y_pred)
clean_on_clean, y_pred_clean = eval_classifier(
    art_clean,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

# save clean model
if SAVE_MODELS:
    save_art_model_state(art_clean, ARTIFACT_DIR / "nn_clean_model.weights.h5")


Training clean model with: {'hidden_layer_sizes': (128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.01, 'batch_size': 64, 'activation': 'relu', 'solver': 'adam', 'learning_rate': 'adaptive', 'max_iter': 1000, 'early_stopping': True, 'n_iter_no_change': 20, 'random_state': 100}

[clean_model_on_clean_test] acc=0.9447 f1=0.8647
confusion matrix:
[[1554   59]
 [  53  358]]
              precision    recall  f1-score   support

           0     0.9670    0.9634    0.9652      1613
           1     0.8585    0.8710    0.8647       411

    accuracy                         0.9447      2024
   macro avg     0.9128    0.9172    0.9150      2024
weighted avg     0.9450    0.9447    0.9448      2024



In [6]:
# Step 2: initialize DeepFool and generate adversarial samples
def make_deepfool_attack(classifier):
    try:
        return DeepFool(
            classifier=classifier,
            max_iter=DEEPFOOL_MAX_ITER,
            epsilon=DEEPFOOL_EPSILON,
            nb_grads=DEEPFOOL_NB_GRADS,
            batch_size=DEEPFOOL_BATCH_SIZE,
            verbose=False,
        )
    except TypeError:
        return DeepFool(
            estimator=classifier,
            max_iter=DEEPFOOL_MAX_ITER,
            epsilon=DEEPFOOL_EPSILON,
            nb_grads=DEEPFOOL_NB_GRADS,
            batch_size=DEEPFOOL_BATCH_SIZE,
            verbose=False,
        )

deepfool = make_deepfool_attack(art_clean)

X_train_adv = np.clip(deepfool.generate(x=X_train), 0.0, 1.0).astype(np.float32)
X_test_adv = np.clip(deepfool.generate(x=X_test), 0.0, 1.0).astype(np.float32)

print("Adversarial data generated with DeepFool:")
print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# If y_train_adv is needed, predict it.
# The handwritten pipeline notes this as optional.
y_train_adv_pred = predict_labels(art_clean, X_train_adv)
y_test_adv_pred = predict_labels(art_clean, X_test_adv)

# For retraining and evaluation targets, keep the original ground-truth labels.
y_train_adv = y_train.copy()
y_test_adv = y_test.copy()

# Build combined clean + adversarial datasets.
X_train_combined = np.concatenate([X_train, X_train_adv], axis=0).astype(np.float32)
y_train_combined = np.concatenate([y_train, y_train_adv], axis=0).astype(np.int64)

X_test_combined = np.concatenate([X_test, X_test_adv], axis=0).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv], axis=0).astype(np.int64)

print("Combined datasets created:")
print("X_train_combined:", X_train_combined.shape)
print("y_train_combined:", y_train_combined.shape)
print("X_test_combined:", X_test_combined.shape)
print("y_test_combined:", y_test_combined.shape)


Adversarial data generated with DeepFool:
X_train_adv: (674, 18)
X_test_adv: (2024, 18)
Combined datasets created:
X_train_combined: (1348, 18)
y_train_combined: (1348,)
X_test_combined: (4048, 18)
y_test_combined: (4048,)


In [7]:
# Evaluate performance of the clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_classifier(
    art_clean,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_classifier(
    art_clean,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)



[clean_model_on_adv_test] acc=0.1616 f1=0.2359
confusion matrix:
[[  65 1548]
 [ 149  262]]
              precision    recall  f1-score   support

           0     0.3037    0.0403    0.0712      1613
           1     0.1448    0.6375    0.2359       411

    accuracy                         0.1616      2024
   macro avg     0.2242    0.3389    0.1535      2024
weighted avg     0.2715    0.1616    0.1046      2024


[clean_model_on_combined_test] acc=0.5531 f1=0.4067
confusion matrix:
[[1619 1607]
 [ 202  620]]
              precision    recall  f1-score   support

           0     0.8891    0.5019    0.6416      3226
           1     0.2784    0.7543    0.4067       822

    accuracy                         0.5531      4048
   macro avg     0.5837    0.6281    0.5241      4048
weighted avg     0.7651    0.5531    0.5939      4048



In [ ]:
# Retrain using ART's AdversarialTrainer with the DeepFool attack
art_adv = make_art_classifier(d_in=X_train.shape[1])
deepfool_for_training = make_deepfool_attack(art_adv)

adv_trainer = AdversarialTrainer(
    classifier=art_adv,
    attacks=deepfool_for_training,
    ratio=ADV_RATIO,  
)

adv_trainer.fit(
    X_train_balanced,
    y_train_balanced_oh,
    batch_size=BATCH_SIZE,
    nb_epochs=NB_EPOCHS,
)

if SAVE_MODELS:
    save_art_model_state(art_adv, ARTIFACT_DIR / "nn_adversarial_trained_model.weights.h5")


Adversarial training epochs:   3%|▎         | 26/1000 [03:24<2:10:47,  8.06s/it]

In [ ]:
# Test using X_test_adv and the combined clean+adv test set
adv_trained_on_adv, y_pred_adv = eval_classifier(
    art_adv,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

# Also check whether adversarial training preserved clean performance
adv_trained_on_clean, y_pred_clean_adv_model = eval_classifier(
    art_adv,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_classifier(
    art_adv,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)


In [ ]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_clean,
    adv_trained_on_adv,
    adv_trained_on_combined,
])

summary_df


In [ ]:
# Save metrics
summary_path = RESULTS_DIR / "nn_deepfool_pipeline_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")


## Dual-Stream Consistency Gate

This section now mirrors the **dual-stream detector flow** used in the Transformer, LSTM, and CNN notebooks.

### What changed
- Treat the clean model as the **Nominal Model**
- Treat the adversarially trained model as the **Guardian Model**
- Detect possible attacks using:
  1. **Prediction disagreement**
  2. **High-confidence disagreement**
  3. A guarded case where the nominal model predicts benign and the guardian predicts anomaly

### Why this is better
The earlier gate converted both model outputs into three custom labels (`Anomaly`, `AttackFlag`, `Nominal`), which did not match the other three pipelines. This version evaluates the gate as an **attack detector**, so you can report:
- **FPR** on clean samples
- **TPR** on adversarial samples
- **F1** for clean-vs-attack detection

It also returns the **final gated prediction**, where the guardian prediction is used when an attack is detected and the nominal prediction is used otherwise.


In [ ]:

# Dual-stream consistency gate aligned with the CNN / LSTM / Transformer notebooks

CONFIDENCE_THRESHOLD = 0.60
DISAGREEMENT_THRESHOLD = 0.55

def predict_with_confidence(art_clf, X: np.ndarray):
    probs = art_clf.predict(X).astype(np.float32)
    preds = np.argmax(probs, axis=1)
    return preds, probs

class DualStreamDetector:
    """
    Dual-stream detector for the NN pipeline.

    Components:
    - Nominal model: trained only on clean data
    - Guardian model: adversarially trained model
    - Consistency gate: flags likely attacks based on disagreement
    """

    def __init__(self, nominal_model, guardian_model):
        self.nominal_model = nominal_model
        self.guardian_model = guardian_model

    def detect_attacks(
        self,
        X: np.ndarray,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        disagreement_threshold: float = DISAGREEMENT_THRESHOLD,
    ):
        preds_nominal, probs_nominal = predict_with_confidence(self.nominal_model, X)
        preds_guardian, probs_guardian = predict_with_confidence(self.guardian_model, X)

        n_samples = len(X)
        flags = np.zeros(n_samples, dtype=int)
        details = []

        for i in range(n_samples):
            yA = int(preds_nominal[i])
            yB = int(preds_guardian[i])
            pA = probs_nominal[i]
            pB = probs_guardian[i]

            conf_nominal = float(pA[yA])
            conf_guardian = float(pB[yB])

            detected = False
            reasons = []

            # Rule 1: any prediction disagreement is suspicious
            if yA != yB:
                detected = True
                reasons.append("disagreement")

            # Rule 2: nominal says benign, guardian says anomaly, both are confident,
            # and the guardian anomaly probability is sufficiently larger
            if yA == 0 and yB == 1:
                if conf_nominal >= confidence_threshold and conf_guardian >= confidence_threshold:
                    prob_diff = float(pB[1] - pA[1])
                    if prob_diff >= disagreement_threshold:
                        detected = True
                        reasons.append("high_confidence_disagreement")
                else:
                    prob_diff = float(pB[1] - pA[1])
            else:
                prob_diff = float(pB[1] - pA[1])

            flags[i] = int(detected)
            details.append({
                "nominal_pred": yA,
                "guardian_pred": yB,
                "nominal_conf": conf_nominal,
                "guardian_conf": conf_guardian,
                "nominal_anom_prob": float(pA[1]),
                "guardian_anom_prob": float(pB[1]),
                "prob_diff_anomaly": prob_diff,
                "detected": bool(detected),
                "reason": ",".join(reasons) if reasons else "none",
            })

        final_preds = np.where(flags == 1, preds_guardian, preds_nominal)
        details_df = pd.DataFrame(details)
        return final_preds, flags, details_df

def evaluate_dual_stream_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }

detector = DualStreamDetector(
    nominal_model=art_clean,
    guardian_model=art_adv,
)

# Evaluate final gated predictions on clean, adversarial, and combined sets
gated_clean_pred, flags_clean, gate_clean_df = detector.detect_attacks(X_test)
gated_adv_pred, flags_adv, gate_adv_df = detector.detect_attacks(X_test_adv)
gated_combined_pred, flags_combined, gate_combined_df = detector.detect_attacks(X_test_combined)

gated_clean_metrics = evaluate_dual_stream_predictions(
    y_test, gated_clean_pred, "dual_stream_final_predictions_on_clean_test"
)
gated_adv_metrics = evaluate_dual_stream_predictions(
    y_test_adv, gated_adv_pred, "dual_stream_final_predictions_on_adv_test"
)
gated_combined_metrics = evaluate_dual_stream_predictions(
    y_test_combined, gated_combined_pred, "dual_stream_final_predictions_on_combined_test"
)

dual_stream_prediction_summary_df = pd.DataFrame([
    gated_clean_metrics,
    gated_adv_metrics,
    gated_combined_metrics,
])

print("\nDual-stream final prediction summary:")
display(dual_stream_prediction_summary_df)

# Evaluate the gate specifically as an attack detector
y_attack_true = np.concatenate([
    np.zeros(len(flags_clean), dtype=int),
    np.ones(len(flags_adv), dtype=int),
])
y_attack_pred = np.concatenate([flags_clean, flags_adv])

dual_stream_detection_results = pd.DataFrame([{
    "attack": "DeepFool",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "disagreement_threshold": DISAGREEMENT_THRESHOLD,
    "FPR": float(flags_clean.mean()),
    "TPR": float(flags_adv.mean()),
    "F1": float(f1_score(y_attack_true, y_attack_pred, zero_division=0)),
    "clean_model_acc_on_adv": float(clean_on_adv["acc"]),
    "guardian_model_acc_on_clean": float(adv_trained_on_clean["acc"]),
    "guardian_model_acc_on_adv": float(adv_trained_on_adv["acc"]),
}])

print("\nDual-stream attack-detection summary:")
display(dual_stream_detection_results)

print("\nGate reason counts on clean test:")
display(gate_clean_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))

print("\nGate reason counts on adversarial test:")
display(gate_adv_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))


def print_prediction_distribution(name: str, y_true: np.ndarray, y_pred: np.ndarray):
    true_dist = dict(zip(*np.unique(y_true.astype(int), return_counts=True)))
    pred_dist = dict(zip(*np.unique(y_pred.astype(int), return_counts=True)))
    print(f"{name} true label dist: {true_dist}")
    print(f"{name} predicted label dist: {pred_dist}")

print("\nPrediction distribution checks:")
print_prediction_distribution("clean gated", y_test, gated_clean_pred)
print_prediction_distribution("adv gated", y_test_adv, gated_adv_pred)
print_prediction_distribution("combined gated", y_test_combined, gated_combined_pred)


In [ ]:
# Save dual-stream outputs
dual_stream_prediction_summary_path = RESULTS_DIR / "nn_dual_stream_prediction_summary.csv"
dual_stream_detection_path = RESULTS_DIR / "nn_dual_stream_detection_results.csv"

dual_stream_prediction_summary_df.to_csv(dual_stream_prediction_summary_path, index=False)
dual_stream_detection_results.to_csv(dual_stream_detection_path, index=False)

gate_clean_df.to_csv(RESULTS_DIR / "nn_dual_stream_gate_clean_details.csv", index=False)
gate_adv_df.to_csv(RESULTS_DIR / "nn_dual_stream_gate_adv_details.csv", index=False)
gate_combined_df.to_csv(RESULTS_DIR / "nn_dual_stream_gate_combined_details.csv", index=False)

print(f"Saved: {dual_stream_prediction_summary_path}")
print(f"Saved: {dual_stream_detection_path}")
